# Local CNN-Transformer Training and Backtest (No Colab)

Runs training/testing locally and exports buy-only portfolio outputs to `outputs/predictions/only_buy`.

In [ ]:
import os
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score


# Ensure project root is on sys.path when notebook is run from notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
if not (project_root / 'src').exists():
    raise RuntimeError(f"Could not locate project root from cwd={cwd}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
print('project_root:', project_root)

from src.colab_data import load_sharded_arrays
from src.train import TrainingConfig, train_model
from src.predict import build_portfolio_performance, build_top_k_portfolio, build_weekly_prediction_table, export_prediction_outputs

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))

In [ ]:
arrays = load_sharded_arrays('data/colab')
metadata = pd.read_parquet('data/colab/model_dataset_metadata.parquet')
vnindex_comparison = pd.read_csv('outputs/tables/vnindex_label_comparison.csv')

expected_label_names = {'Avoid', 'Hold', 'Buy'}
actual_label_names = set(metadata['label_name'].dropna().unique())
if actual_label_names != expected_label_names:
    raise RuntimeError(f'Expected labels {sorted(expected_label_names)} but got {sorted(actual_label_names)}')

metadata['rebalance_date'] = pd.to_datetime(metadata['rebalance_date']).dt.normalize()
metadata['next_rebalance_date'] = pd.to_datetime(metadata['next_rebalance_date']).dt.normalize()
vnindex_comparison['rebalance_date'] = pd.to_datetime(vnindex_comparison['rebalance_date']).dt.normalize()
vnindex_comparison['next_rebalance_date'] = pd.to_datetime(vnindex_comparison['next_rebalance_date']).dt.normalize()

print({k: v.shape for k, v in arrays.items() if hasattr(v, 'shape')})
print('metadata_rows:', len(metadata))

In [ ]:
config = TrainingConfig(
    epochs=15,
    batch_size=512 if torch.cuda.is_available() else 128,
    learning_rate=1e-3,
    patience=4,
    checkpoint_dir='checkpoints'
)
model, metrics = train_model(arrays, config)
print('best_epoch:', metrics['best_epoch'])
print('accuracy:', metrics['accuracy'])
print('macro_f1:', metrics['macro_f1'])
print('checkpoint:', metrics['checkpoint_path'])

In [ ]:
history = metrics['history']
epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_accuracy = [row['val_accuracy'] for row in history]
val_macro_f1 = [row['val_macro_f1'] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, train_loss, marker='o')
axes[0].set_title('Training Loss')
axes[1].plot(epochs, val_accuracy, marker='o', label='Val Accuracy')
axes[1].plot(epochs, val_macro_f1, marker='o', label='Val Macro F1')
axes[1].set_title('Validation Metrics')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
X_all = np.concatenate([arrays['X_train'], arrays['X_val'], arrays['X_test']], axis=0)
dataset_all = TensorDataset(torch.from_numpy(X_all).float())
loader_all = DataLoader(dataset_all, batch_size=2048, shuffle=False)

model.eval()
all_probs_list = []
with torch.no_grad():
    for (xb,) in loader_all:
        logits = model(xb.to(next(model.parameters()).device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs_list.append(probs)
all_probs = np.vstack(all_probs_list)

prediction_table = build_weekly_prediction_table(metadata, all_probs).copy()
prediction_table['score'] = prediction_table['p_buy']
prediction_table['rank'] = prediction_table.groupby('rebalance_date')['score'].rank(method='first', ascending=False).astype(int)

TOP_K = 5
top_table = prediction_table[prediction_table['rank'] <= TOP_K].copy()
top_table['portfolio_weight'] = 1.0 / TOP_K

output_dir = 'outputs/predictions/only_buy'
os.makedirs(output_dir, exist_ok=True)
score_path, top_path = export_prediction_outputs(prediction_table, output_dir, top_k=TOP_K, buy_only=False)

print('score_csv:', score_path)
print('top_csv:', top_path)

In [ ]:
fee_rates = [0.0, 0.0015, 0.0025, 0.0035]
perf_tables = []
summary_rows = []

for fee_rate in fee_rates:
    perf = build_portfolio_performance(top_table, vnindex_comparison=vnindex_comparison, fee_rate=fee_rate)
    perf['fee_rate'] = fee_rate
    perf_tables.append(perf)

    test_perf_fee = perf[perf['rebalance_date'] >= pd.Timestamp('2024-01-01')].copy()
    if len(test_perf_fee) == 0:
        continue

    port_std = test_perf_fee['portfolio_simple_return'].std()
    port_sharpe = np.sqrt(52) * test_perf_fee['portfolio_simple_return'].mean() / port_std if port_std > 0 else np.nan

    vn_sharpe = np.nan
    if 'vnindex_simple_return' in test_perf_fee.columns and test_perf_fee['vnindex_simple_return'].notna().any():
        vn_std = test_perf_fee['vnindex_simple_return'].std()
        vn_sharpe = np.sqrt(52) * test_perf_fee['vnindex_simple_return'].mean() / vn_std if vn_std > 0 else np.nan

    summary_rows.append({
        'fee_rate': fee_rate,
        'weeks_test': int(len(test_perf_fee)),
        'portfolio_total_return_test': float((1.0 + test_perf_fee['portfolio_simple_return']).prod() - 1.0),
        'portfolio_sharpe_test': float(port_sharpe) if pd.notna(port_sharpe) else np.nan,
        'vnindex_sharpe_test': float(vn_sharpe) if pd.notna(vn_sharpe) else np.nan,
        'avg_turnover_test': float(test_perf_fee['portfolio_turnover'].mean()) if 'portfolio_turnover' in test_perf_fee.columns else np.nan,
        'avg_trading_cost_test': float(test_perf_fee['trading_cost'].mean()) if 'trading_cost' in test_perf_fee.columns else np.nan,
    })

perf_all = pd.concat(perf_tables, ignore_index=True)
summary_df = pd.DataFrame(summary_rows).sort_values('fee_rate').reset_index(drop=True)

base_fee_rate = 0.0025
perf_base = perf_all[perf_all['fee_rate'] == base_fee_rate].copy()
performance_csv = os.path.join(output_dir, 'weekly_top_5_v2_full_reset_performance.csv')
summary_csv = os.path.join(output_dir, 'weekly_top_5_v2_full_reset_cost_sensitivity.csv')
perf_base.to_csv(performance_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print('performance_csv:', performance_csv)
print('cost_sensitivity_csv:', summary_csv)
display(summary_df)

In [ ]:
test_perf = perf_base[perf_base['rebalance_date'] >= pd.Timestamp('2024-01-01')].copy()
test_perf['portfolio_cumulative_test'] = (1.0 + test_perf['portfolio_simple_return']).cumprod()
if 'vnindex_simple_return' in test_perf.columns and test_perf['vnindex_simple_return'].notna().any():
    test_perf['vnindex_cumulative_test'] = (1.0 + test_perf['vnindex_simple_return']).cumprod()

required_cols = {'rebalance_date', 'portfolio_cumulative_test'}
missing = [c for c in required_cols if c not in test_perf.columns]
if missing:
    raise ValueError(f"test_perf is missing required columns: {missing}. Available: {test_perf.columns.tolist()}")

plot_df = test_perf.copy()
plot_df['rebalance_date'] = pd.to_datetime(plot_df['rebalance_date'], errors='coerce')
plot_df = plot_df.dropna(subset=['rebalance_date', 'portfolio_cumulative_test']).sort_values('rebalance_date')

fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(plot_df['rebalance_date'], plot_df['portfolio_cumulative_test'], label='Top 5 Weekly Reset Portfolio (Test)', linewidth=2.2)

if 'vnindex_cumulative_test' in plot_df.columns:
    vn = plot_df.dropna(subset=['vnindex_cumulative_test'])
    if not vn.empty:
        ax.plot(vn['rebalance_date'], vn['vnindex_cumulative_test'], label='VNINDEX (Test)', linewidth=2.0)

ax.set_title('Top 5 Weekly Reset Portfolio vs VNINDEX - Test Period Only')
ax.set_xlabel('Rebalance Date')
ax.set_ylabel('Cumulative Growth of 1.0')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()